# Evolution-1 · PPO baseline on Colab

Runs `experiments/exp6_ppo.json`: GA vs PPO, 5 seeds each, 25,000 episodes = 15M environment steps
per run, on the same 15->16->4 tanh brain.

**Read this before you start**

- PPO here is **CPU-bound, not GPU-bound**. The policy is a 16-unit MLP (324 weights) and every
  step is Pymunk physics + Python, so a GPU does nothing. Use **Runtime -> Change runtime type ->
  CPU**. A GPU runtime is not faster; it usually comes with a smaller CPU allocation.
- Measured throughput on a laptop core: **~680 steps/s**, i.e. **~6 h per PPO seed**. Seeds run in
  parallel, one process each, so 5+ vCPU is ~6-7 h wall clock. Colab's free tier is 2 vCPU, which
  is ~15 h and will not fit in one session.
- Free Colab disconnects. The run cell syncs every finished seed to Google Drive, so a disconnect
  costs at most the seed in flight: reconnect, re-run cell 2 and cell 5, and it resumes.
- A half-finished seed is deleted and restarted (the runner's resume contract), so do not
  oversubscribe `--workers` beyond the vCPU count.

In [ ]:
REPO = "https://github.com/000wahab000/Evolution.git"
BRANCH = "ppo-baseline"

%cd /content
!rm -rf Evolution
!git clone -q --depth 1 --branch {BRANCH} {REPO} Evolution
%cd /content/Evolution
!pip install -q -e ".[rl]"
!python -c "import os, torch, stable_baselines3, src.rl.train; print('torch', torch.__version__, '| sb3', stable_baselines3.__version__, '| vCPU', os.cpu_count())"

In [ ]:
# ~2 minutes: proves the install, the env contract (terminated vs truncated, fall penalty)
# and the CreatureBrain export path all work before you spend six hours.
!python scripts/run_experiment.py experiments/smoke_ppo.json --workers 1 --out /content/smoke

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, subprocess, sys, time

OUT = "/content/results"                            # fast local disk
SAVE = "/content/drive/MyDrive/Evolution-results"   # survives disconnects
EXP = "exp6_ppo"
VARIANTS, SEEDS = ("ga", "ppo"), range(5)
LOG = "/content/exp6_ppo.log"

os.makedirs(OUT, exist_ok=True)
os.makedirs(SAVE, exist_ok=True)


def run_dir(root, v, s):
    return f"{root}/{EXP}/{v}/seed_{s}"


def restore():
    """Pull finished seeds back from Drive after a disconnect."""
    for v in VARIANTS:
        for s in SEEDS:
            src, dst = run_dir(SAVE, v, s), run_dir(OUT, v, s)
            if os.path.exists(f"{src}/DONE") and not os.path.exists(f"{dst}/DONE"):
                shutil.copytree(src, dst, dirs_exist_ok=True)
                print(f"restored {v}/seed_{s}")


def sync():
    """Push each finished seed to Drive."""
    for v in VARIANTS:
        for s in SEEDS:
            src, dst = run_dir(OUT, v, s), run_dir(SAVE, v, s)
            if os.path.exists(f"{src}/DONE") and not os.path.exists(f"{dst}/DONE"):
                shutil.copytree(src, dst, dirs_exist_ok=True)
                print(f"saved {v}/seed_{s}")


restore()
workers = max(1, min(5, os.cpu_count() or 2))
print(f"{os.cpu_count()} vCPU -> {workers} seeds in parallel (PPO is ~6 h/seed, GA ~15 min/seed)")

cmd = [sys.executable, "scripts/run_experiment.py", "experiments/exp6_ppo.json",
       "--workers", str(workers), "--out", OUT]
print(" ".join(cmd), "\n")

with open(LOG, "w") as log:
    p = subprocess.Popen(cmd, cwd="/content/Evolution", stdout=log, stderr=subprocess.STDOUT)
    pos = 0
    try:
        while p.poll() is None:
            sync()
            with open(LOG, encoding="utf-8", errors="replace") as f:
                f.seek(pos)
                new = f.read()
                pos = f.tell()
            if new:
                print(new, end="")
            time.sleep(30)
    except KeyboardInterrupt:
        p.terminate()
        print("\ninterrupted")
    finally:
        sync()

finished = sum(os.path.exists(f"{run_dir(OUT, v, s)}/DONE") for v in VARIANTS for s in SEEDS)
print("exit code:", p.returncode)
print(f"finished runs: {finished}/10")

In [ ]:
# Only meaningful once all 10 runs are DONE; verify_experiment inside analyze will say so.
!python scripts/analyze.py /content/results/exp6_ppo

import os, shutil
for name in ("REPORT.md", "stats.json"):
    src = f"/content/results/exp6_ppo/{name}"
    if os.path.exists(src):
        shutil.copy(src, f"/content/drive/MyDrive/Evolution-results/exp6_ppo/{name}")
        print("saved", name)